In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

#import personnal tools
import sys
sys.path.append('../tools/')
from info import *
from tools_generic import *

# Load files

In [ ]:
site_list=["d17","d47","d85","dmc"]
start_date = '20241201'
end_date = '20260630'

In [ ]:
data = {}
daily_data = {}

In [ ]:
# data, daily_data = create_dict_data_datadaily(sites, sensors, start_date, end_date)
data = create_data(sites, sensors, start_date, end_date)

In [ ]:
# open wind_beginning files
var_list=["wspd1","wspd2","wspd3", "wdir"]
for site in site_list:
    file_path=f'../../data/WIND_BEGINNING_{site}_20241201_20250313.netcdf'
    
    ds = xr.open_dataset(file_path, engine='netcdf4')
   
    for var in var_list:
        # Check if the variable exists in both datasets
        if var in data["WIND"][site] and var in ds:
            wind_var = data["WIND"][site][var]
            ds_var = ds[var]

            # Align the datasets along the time dimension
            wind_var_aligned, ds_var_aligned = xr.align(wind_var, ds_var, join='outer')

            # Merge the datasets: prioritize non-NaN values from wind_var, fall back to ds_var
            merged_var = wind_var_aligned.where(~np.isnan(wind_var_aligned), ds_var_aligned)

            # Update the data["WIND"][site][var] with the merged result
            data["WIND"][site][var] = merged_var
        else:
            print(f"Variable {var} not found in both datasets for site {site}")

    ds.close()  # Close the dataset to free resources

In [ ]:
# create daily
daily_data = create_daily_data(data)

In [ ]:
# Extract period of interest (golden month)
# golden = filter_datasets_golden(
#     data, start_date="2025-02-01", end_date="2025-02-28"
#  )
# #golden

## Filter outliers

In [ ]:
varlist=['snowflux']
data = filter_data_by_max_values(
    data, 
    varlist
    )

# Stats

In [ ]:
variable = 'FluxMean1'
compute_variable_stats(golden, variable)

In [ ]:
var="FluxMean1"
plot_binned_distribution(golden, var, variable_to_sensor, bin_number=30, min_value=0, max_value=200)

In [ ]:
var="snowflux"
plot_binned_distribution(golden, var, variable_to_sensor, bin_number=30, min_value=0, max_value=200)

# Plotting

In [ ]:
variables=["FluxMean1", "FluxMean2", 'snowflux', 'Hagl', 'wspd1','wspd2', ]
variables = ['wspd1','wspd2']
# variables = ["FluxMean1",'snowflux', 'Hagl'] 
# variables = ['T1','T2','T3']
variables=['Hagl']

plot_per_var_multiple_sites(
    # sensor_datasets=daily_data,
    sensor_datasets=data,
    variables = variables,
    # sites=sites,
    sites=["d17","d47"],
    figsize=(15, 5),
    # ymin=[-80, -80, -80],  # Custom ymin for each variable
    # ymax=[500, 500, 500], # Custom ymax for each variable
    ymin=[-0.1],ymax=[3],
    # colors=["blue", "red"],  # Valid color strings (e.g., hex or named colors)
)

In [ ]:
event = {"number_of_timesteps": 24,
             "threshold": 10}
# Call the function
event_masks = detect_and_plot_events(golden, "FluxMean1", event, plot=True, variable_to_sensor=variable_to_sensor)

In [ ]:
variables=["FluxMean1", "FluxMean2", 'snowflux', 'Hagl' ,'wspd1','wspd2', ]
# variables = ['wspd1','wspd2']
# variables = ["FluxMean1",'snowflux', 'wspd1']
# variables = ['T1','T2','T3']

plot_monthly_availability_table(
    sensor_datasets=daily_data,
    # sensor_datasets=data,
    variables = variables,
    sites=sites,
    figsize=(16, 8),
    # ymin=[-80, -80, -80],  # Custom ymin for each variable
    # ymax=[500, 500, 500], # Custom ymax for each variable
    # colors=["blue", "red"],  # Valid color strings (e.g., hex or named colors)
)